## Tarea 2: API REST — Base de datos de videojuegos RAWG
Extraer, analizar y comparar datos de videojuegos usando Python.

### Configuración inicial

In [3]:
import requests
import pandas as pd
import os

API_KEY = "befbcf31701645988a35aaf363441d92"
BASE_URL = "https://api.rawg.io/api"

### Clase auxiliar para las consultas en la API


In [4]:
class RAWGClient:
    """Cliente para consumir la API de RAWG con conteo de requests."""
    
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = "https://api.rawg.io/api"
        self.total_requests = 0
    
    def get(self, endpoint, params=None):
        """Realiza una solicitud GET a la API de RAWG."""
        if params is None:
            params = {}
        params["key"] = self.api_key
        
        url = f"{self.base_url}/{endpoint}"
        response = requests.get(url, params=params)
        self.total_requests += 1
        
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Error {response.status_code}: {response.text}")
            return None
    
    def resumen_requests(self):
        """Muestra el total de solicitudes realizadas a la API."""
        print(f"Total de solicitudes realizadas a la API: {self.total_requests}")
        return self.total_requests

# Crear instancia del cliente
client = RAWGClient(API_KEY)
print("Cliente RAWG configurado correctamente.")

Cliente RAWG configurado correctamente.


## Parte A - Exploración general

#### ¿Cuántos juegos tiene registrados RAWG en total?

In [6]:
data = client.get("games", params={"page_size": 1})

if data:
    total_juegos = data["count"]
    print(f"RAWG tiene registrados un total de {total_juegos:,} juegos en su base de datos.")
else:
    print("No se pudo obtener la información.")

RAWG tiene registrados un total de 898,502 juegos en su base de datos.


##  Parete B - Análisis de categoría
#### ¿Cuáles son los 5 juegos mejor valorados de todos los tiempos según Metacritic?

In [8]:
data = client.get("games", params={
    "ordering": "-metacritic",
    "page_size": 5
})

if data:
    juegos_b1 = []
    for game in data["results"]:
        juegos_b1.append({
            "Nombre": game["name"],
            "Rating": game.get("rating", "N/A"),
            "Metacritic": game.get("metacritic", "N/A")
        })
    
    df_b1 = pd.DataFrame(juegos_b1)
    print("El top 5 de juegos mejor valorados según Metacritic:\n")
    print(df_b1.to_string(index=False))

El top 5 de juegos mejor valorados según Metacritic:

                              Nombre  Rating  Metacritic
The Legend of Zelda: Ocarina of Time    4.38          99
                  Soulcalibur (1998)    0.00          98
                         Soulcalibur    4.38          98
                   Baldur's Gate III    4.44          97
                       Metroid Prime    4.35          97


#### ¿Cuáles son los 10 mejores juegos disponibles en Steam (store_id=1)?

In [9]:
data = client.get("games", params={
    "stores": 1,  # Steam
    "ordering": "-metacritic",
    "page_size": 10
})

if data:
    juegos_b2 = []
    for game in data["results"]:
        juegos_b2.append({
            "Nombre": game["name"],
            "Rating": game.get("rating", "N/A"),
            "Metacritic": game.get("metacritic", "N/A")
        })
    
    df_b2 = pd.DataFrame(juegos_b2)
    print("Top 10 juegos en Steam según Metacritic:\n")
    print(df_b2.to_string(index=False))

Top 10 juegos en Steam según Metacritic:

                               Nombre  Rating  Metacritic
                    Baldur's Gate III    4.44          97
                  Half-Life 2: Update    4.13          96
                            Half-Life    4.38          96
                Red Dead Redemption 2    4.59          96
                          Half-Life 2    4.48          96
                             BioShock    4.36          96
Grand Theft Auto IV: Complete Edition    4.57          95
             Divinity: Original Sin 2    4.38          95
                             Portal 2    4.58          95
                  Red Dead Redemption    4.42          95


## Parte C - Comparaciones
#### C1: Compara los 5 mejores juegos de PC (platform_id=4) con los 5 mejores de PS5 (platform_id=187).
##### ¿Qué plataforma tiene los juegos mejor valorados?

In [10]:
# Top 5 PC
data_pc = client.get("games", params={
    "platforms": 4,
    "ordering": "-metacritic",
    "page_size": 5
})

# Top 5 PS5
data_ps5 = client.get("games", params={
    "platforms": 187,
    "ordering": "-metacritic",
    "page_size": 5
})

if data_pc and data_ps5:
    juegos_pc = []
    for game in data_pc["results"]:
        juegos_pc.append({
            "Nombre": game["name"],
            "Rating": game.get("rating", 0),
            "Metacritic": game.get("metacritic", 0),
            "Plataforma": "PC"
        })
    
    juegos_ps5 = []
    for game in data_ps5["results"]:
        juegos_ps5.append({
            "Nombre": game["name"],
            "Rating": game.get("rating", 0),
            "Metacritic": game.get("metacritic", 0),
            "Plataforma": "PS5"
        })
    
    df_pc = pd.DataFrame(juegos_pc)
    df_ps5 = pd.DataFrame(juegos_ps5)
    
    print("Top 5 juegos de PC:\n")
    print(df_pc.to_string(index=False))
    
    print("\nTop 5 juegos de PS5:\n")
    print(df_ps5.to_string(index=False))
    
    # Comparación
    avg_pc = df_pc["Rating"].mean()
    avg_ps5 = df_ps5["Rating"].mean()
    avg_meta_pc = df_pc["Metacritic"].mean()
    avg_meta_ps5 = df_ps5["Metacritic"].mean()
    
    print(f"\n Comparación de promedios:")
    print(f"   PC  → Rating: {avg_pc:.2f} | Metacritic: {avg_meta_pc:.1f}")
    print(f"   PS5 → Rating: {avg_ps5:.2f} | Metacritic: {avg_meta_ps5:.1f}")
    
    ganador = "PC" if avg_meta_pc > avg_meta_ps5 else "PS5"
    print(f"\n La plataforma con mejores juegos valorados es: {ganador}")

Top 5 juegos de PC:

               Nombre  Rating  Metacritic Plataforma
    Baldur's Gate III    4.44          97         PC
  Half-Life 2: Update    4.13          96         PC
            Half-Life    4.38          96         PC
Red Dead Redemption 2    4.59          96         PC
          Half-Life 2    4.48          96         PC

Top 5 juegos de PS5:

                     Nombre  Rating  Metacritic Plataforma
          Baldur's Gate III    4.44          97        PS5
        Red Dead Redemption    4.42          95        PS5
                 Elden Ring    4.38          95        PS5
The Elder Scrolls V: Skyrim    4.42          94        PS5
                      Quake    4.25          94        PS5

 Comparación de promedios:
   PC  → Rating: 4.40 | Metacritic: 96.2
   PS5 → Rating: 4.38 | Metacritic: 95.0

 La plataforma con mejores juegos valorados es: PC


#### C2: Elige 3 juegos famosos y crea una tabla comparativa con: nombre, calificación, Metacritic, géneros y plataformas.

In [ ]:
juegos_famosos = ["Minecraft", "God of War", "Among Us"]
comparativa = []

for nombre_juego in juegos_famosos:
    data = client.get("games", params={
        "search": nombre_juego,
        "page_size": 1
    })
    
    if data and data["results"]:
        game = data["results"][0]
        generos = ", ".join([g["name"] for g in game.get("genres", [])])
        plataformas = ", ".join([p["platform"]["name"] for p in game.get("platforms", [])])
        
        comparativa.append({
            "Nombre": game["name"],
            "Rating": game.get("rating", "N/A"),
            "Metacritic": game.get("metacritic", "N/A"),
            "Géneros": generos,
            "Plataformas": plataformas
        })

df_comp = pd.DataFrame(comparativa)
print("Tabla comparativa de 3 juegos famosos:\n")
print(df_comp.to_string(index=False))

Tabla comparativa de 3 juegos famosos:

      Nombre  Rating  Metacritic                                                  Géneros                                                                                                                     Plataformas
   Minecraft    4.44          83 Action, Arcade, Simulation, Indie, Massively Multiplayer PC, Xbox One, PlayStation 4, Nintendo Switch, iOS, Android, Nintendo 3DS, macOS, Linux, Xbox 360, PlayStation 3, PS Vita, Wii U
God of War I    4.36          94                                                   Action                                                                                           PlayStation 3, PlayStation 2, PS Vita
    Among Us    3.83          82                               Casual, Action, Simulation                                                       PC, PlayStation 5, PlayStation 4, Xbox One, Nintendo Switch, iOS, Android


#### C3: Consulta los 5 mejores juegos de al menos 4 géneros diferentes , calcula la calificación promedio de cada uno y determina qué género produce los mejores juegos según los usuarios.

In [18]:
# Géneros: Action=4, RPG=5, Puzzle=7, Strategy=10
generos_info = {
    "Action": 4,
    "RPG": 5, 
    "Puzzle": 7,
    "Strategy": 10
}

resultados_generos = []

for nombre_genero, genre_id in generos_info.items():
    data = client.get("games", params={
        "genres": genre_id,
        "ordering": "-rating",
        "page_size": 5
    })
    
    if data and data["results"]:
        ratings = [g.get("rating", 0) for g in data["results"]]
        promedio = sum(ratings) / len(ratings)
        
        print(f"\n Top 5 juegos de {nombre_genero}:")
        for g in data["results"]:
            print(f"   - {g['name']} (Rating: {g.get('rating', 'N/A')})")
        
        resultados_generos.append({
            "Género": nombre_genero,
            "Rating Promedio": round(promedio, 2)
        })

df_generos = pd.DataFrame(resultados_generos)
print("\n Rating promedio por género:\n")
print(df_generos.to_string(index=False))

mejor_genero = df_generos.loc[df_generos["Rating Promedio"].idxmax()]
print(f"\n El género que produce los mejores juegos según los usuarios es: {mejor_genero['Género']} ({mejor_genero['Rating Promedio']})")


 Top 5 juegos de Action:
   - The Elder Scrolls VI (Rating: 4.86)
   - Super Robot Taisen: Original Generation (Rating: 4.83)
   - Mass Effect Trilogy (Rating: 4.75)
   - Godzilla Save the Earth (Rating: 4.71)
   - Bloodborne: The Old Hunters (Rating: 4.71)

 Top 5 juegos de RPG:
   - The Elder Scrolls VI (Rating: 4.86)
   - The Witcher 3: Wild Hunt – Blood and Wine (Rating: 4.81)
   - The Witcher 3 Wild Hunt - Complete Edition (Rating: 4.8)
   - The Witcher 3: Wild Hunt – Hearts of Stone (Rating: 4.76)
   - Mass Effect Trilogy (Rating: 4.75)

 Top 5 juegos de Puzzle:
   - That level again 2 (Rating: 4.67)
   - Death Palette (Rating: 4.67)
   - One Night at Flumpty's (Rating: 4.67)
   - AirXonix (Rating: 4.6)
   - Portal 2 (Rating: 4.58)

 Top 5 juegos de Strategy:
   - Super Robot Taisen: Original Generation (Rating: 4.83)
   - Victoria 2: Heart of Darkness (Rating: 4.71)
   - Shining Force III (Rating: 4.67)
   - Championship Manager Season 03/04 (Rating: 4.67)
   - Gemfire (Rating:

#### C4: Compara los mejores juegos de 3 años diferentes a tu elección.
##### ¿En qué año se lanzaron los juegos con la puntuación media más alta en Metacritic?

In [19]:
# 2015: prepandemia 2020: durante la pandemia  2023: post pandemia
anios = ["2015", "2020", "2023"]
resultados_anios = []

for anio in anios:
    data = client.get("games", params={
        "dates": f"{anio}-01-01,{anio}-12-31",
        "ordering": "-metacritic",
        "page_size": 5
    })
    
    if data and data["results"]:
        metacritics = [g.get("metacritic", 0) for g in data["results"] if g.get("metacritic")]
        promedio = sum(metacritics) / len(metacritics) if metacritics else 0
        
        print(f"\n Top 5 juegos de {anio}:")
        for g in data["results"]:
            print(f"   - {g['name']} (Metacritic: {g.get('metacritic', 'N/A')})")
        
        resultados_anios.append({
            "Año": anio,
            "Metacritic Promedio": round(promedio, 2)
        })

df_anios = pd.DataFrame(resultados_anios)
print("\n Metacritic promedio por año:\n")
print(df_anios.to_string(index=False))

mejor_anio = df_anios.loc[df_anios["Metacritic Promedio"].idxmax()]
print(f"\n El año con los juegos mejor puntuados es: {mejor_anio['Año']} (Metacritic promedio: {mejor_anio['Metacritic Promedio']})")



 Top 5 juegos de 2015:
   - Half-Life 2: Update (Metacritic: 96)
   - Divinity: Original Sin - Enhanced Edition (Metacritic: 94)
   - Prune (Metacritic: 92)
   - Bloodborne (Metacritic: 92)
   - Undertale (Metacritic: 92)

 Top 5 juegos de 2020:
   - Persona 5 Royal (Metacritic: 94)
   - Half-Life: Alyx (Metacritic: 93)
   - Hades (Metacritic: 93)
   - The Last of Us Part II (Metacritic: 93)
   - Demon's Souls (2020) (Metacritic: 92)

 Top 5 juegos de 2023:
   - Baldur's Gate III (Metacritic: 97)
   - The Legend of Zelda: Tears of the Kingdom (Metacritic: 96)
   - Sea of Stars (Metacritic: 90)
   - Diablo IV (Metacritic: 90)
   - Gravity Circuit (Metacritic: 89)

 Metacritic promedio por año:

 Año  Metacritic Promedio
2015                 93.2
2020                 93.0
2023                 92.4

 El año con los juegos mejor puntuados es: 2015 (Metacritic promedio: 93.2)


#### C5: Exporta los 20 mejores juegos de todos los tiempos a un archivo CSV con el nombre especificado

In [20]:
data = client.get("games", params={
    "ordering": "-metacritic",
    "page_size": 20
})

if data:
    top20 = []
    for game in data["results"]:
        generos = game.get("genres", [])
        main_genre = generos[0]["name"] if generos else "N/A"
        
        top20.append({
            "name": game["name"],
            "rating": game.get("rating", "N/A"),
            "metacritic": game.get("metacritic", "N/A"),
            "release_date": game.get("released", "N/A"),
            "main_genre": main_genre
        })
    
    df_top20 = pd.DataFrame(top20)
        
    os.makedirs("output", exist_ok=True)
    df_top20.to_csv("output/top20_rawg.csv", index=False)
    
    print("Primeras 5 filas:\n")
    print(df_top20.head().to_string(index=False))

Primeras 5 filas:

                                name  rating  metacritic release_date main_genre
The Legend of Zelda: Ocarina of Time    4.38          99   1998-11-21     Action
                  Soulcalibur (1998)    0.00          98   1998-07-30   Fighting
                         Soulcalibur    4.38          98   1998-07-30     Action
                   Baldur's Gate III    4.44          97   2023-08-03  Adventure
                       Metroid Prime    4.35          97   2002-11-17     Action


#### c4: Reflexiones y conclusiones

##### ¿Qué fue lo más interesante que encontraste en los datos?
Me sorprendió que RAWG tenga registrados más de 898,000 juegos. No imaginaba que existiera una base de datos tan grande de videojuegos. También me llamó la atención que un juego de 1998 (Zelda: Ocarina of Time) siga siendo el #1 de todos los tiempos con Metacritic de 99, superando a juegos modernos con gráficos mucho más avanzados.
##### ¿Qué género o plataforma te sorprendió más y por qué?
Me sorprendió que RPG sea el género con el rating promedio más alto (4.80), por encima de Action y Strategy. También que PC tenga mejores promedios que PS5, aunque la diferencia es mínima (96.2 vs 95.0 en Metacritic).
##### ¿Qué otra pregunta le harías a esta API si tuvieras más tiempo?
Me gustaría investigar cuáles son los juegos gratuitos mejor valorados, para saber si la calidad depende del precio o no. También buscaría qué país o estudio produce los juegos con mejor puntuación.

In [21]:
# Total de solicitudes realizadas
client.resumen_requests()

Total de solicitudes realizadas a la API: 38


38